# ML Final Skripsi di Google Colab
**Rahul Ghulam Hidayatulloh — Klasifikasi Kondisi Sel T CD4 pada Infeksi HIV (fitur fisis citra + machine learning)**

Notebook ini menjalankan **seluruh** pengolahan data dan *machine learning* final (rencana: `docs/RENCANA-ML-COLAB.md`):
1. Hubungkan Google Drive (`MyDrive/Project-Skripsi/`): data mentah di `data/raw/`, fitur per sel disimpan ke `data/processed/`.
2. Ambil kode terbaru dari GitHub `raaahl15/Project-Skripsi`.
3. Ekstraksi fitur (dilewati otomatis bila sudah ada di Drive): heiDATA, Illinois (semua sel), Essig exp1–exp4.
4. Latih dan evaluasi tiga model (A hidup/mati, B terinfeksi/tidak, C laten/aktif/ditekan) dengan protokol yang sama.
5. Hasil kecil di-*commit* dan di-*push* ke GitHub (`results/final/`); hasil lengkap + model ke Drive.

**Persiapan sekali saja (Colab → ikon kunci *Secrets* di panel kiri, aktifkan *Notebook access*):**
- `GITHUB_TOKEN_RW` — GitHub → Settings → Developer settings → *Fine-grained tokens* → *Generate new token*: Repository access
  **Only select repositories → Project-Skripsi**, permission **Contents: Read and write**, masa berlaku 30 hari.
- (Opsional) `DRYAD_CLIENT_ID` dan `DRYAD_CLIENT_SECRET` — akun API Dryad, untuk mengunduh Essig exp3/exp4 langsung ke Drive.
- **Jangan menempelkan token di sel notebook atau di chat.** Bila token terlihat orang lain, segera *revoke* dan buat baru.

Jalankan: **Runtime → Run all**. Runtime CPU cukup. Bila sesi terputus, jalankan ulang: ekstraksi yang sudah selesai dilewati,
ekstraksi Essig yang terputus dilanjutkan dari *checkpoint* di Drive.

## 1. Pengaturan run

In [ ]:
RUN_MODELS = ['B', 'C', 'A']           # urutan: B dan C cepat; A menunggu fitur Essig lengkap
ESSIG_EXPS = ['exp1', 'exp2', 'exp3', 'exp4']   # hanya yang zip-nya ada di Drive yang diolah
DOWNLOAD_DRYAD = []                    # mis. ['exp3.zip', 'exp4.zip'] bila token Dryad sudah ada di Secrets
QUICK = False                          # True = uji kode cepat (hasil BUKAN untuk naskah)
PUSH_TO_GITHUB = True
GIT_NAME, GIT_EMAIL = 'Rahul Ghulam Hidayatulloh', 'raaahl15@users.noreply.github.com'

## 2. Runtime dan Google Drive

In [ ]:
!nproc; free -h | head -2; df -h /content | tail -1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/Project-Skripsi')
RAW, PROC = DRIVE / 'data/raw', DRIVE / 'data/processed'
PROC.mkdir(parents=True, exist_ok=True)
for p in sorted(RAW.iterdir()):
    print(f'{p.stat().st_size / 2**30:7.2f} GiB  {p.name}')

## 3. Ambil kode dari GitHub (token dari Colab Secrets)

In [ ]:
import os, subprocess, base64, time
from google.colab import userdata
REPO = '/content/skripsi-ml'
URL = 'https://github.com/raaahl15/Project-Skripsi.git'   # no token in the URL
_tok = userdata.get('GITHUB_TOKEN_RW').strip()
_auth = base64.b64encode(f'x-access-token:{_tok}'.encode()).decode()

def git(*args):
    # token is sent only as an HTTP header for this one command and is masked in every output
    r = subprocess.run(['git', '-c', f'http.extraHeader=Authorization: Basic {_auth}', *args],
                       capture_output=True, text=True, env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    out = (r.stdout + r.stderr).replace(_tok, '***').replace(_auth, '***')
    if r.returncode != 0:
        raise RuntimeError('git gagal (kode %d), pesan (token disensor):\n%s' % (r.returncode, out))
    return out

print(git('clone', URL, REPO) if not os.path.exists(REPO) else git('-C', REPO, 'pull', '--ff-only'))
os.chdir(REPO)
subprocess.run(['git', 'config', 'user.name', GIT_NAME]); subprocess.run(['git', 'config', 'user.email', GIT_EMAIL])
SHA = subprocess.check_output(['git', 'rev-parse', '--short=7', 'HEAD'], text=True).strip()
RUN_ID = time.strftime('%Y%m%d-%H%M', time.gmtime()) + '-' + SHA + ('-quick' if QUICK else '')
print('RUN_ID =', RUN_ID)

## 4. Pustaka (versi yang memengaruhi hasil dikunci)

In [ ]:
!pip install -q -r requirements-colab.txt
!pip freeze > /content/pip_freeze.txt
import sklearn, skimage; print('scikit-learn', sklearn.__version__, '| scikit-image', skimage.__version__)

## 5. (Opsional) Unduh Essig dari Dryad langsung ke Drive

In [ ]:
if DOWNLOAD_DRYAD:
    for k in ('DRYAD_CLIENT_ID', 'DRYAD_CLIENT_SECRET'):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            print('Secret', k, 'tidak ada; mencoba tanpa token')
    !python -m src.final.dryad_download --files {' '.join(DOWNLOAD_DRYAD)} --dest "{RAW}"
else:
    print('Lewati unduhan Dryad')

## 6. Ekstraksi fitur heiDATA (209 inti, 14 bidang pandang)

In [ ]:
import zipfile, shutil
HEI_OUT = PROC / 'heidata'
if (HEI_OUT / 'cells.csv').exists():
    print('sudah ada:', HEI_OUT / 'cells.csv')
else:
    zipfile.ZipFile(RAW / 'dataverse_files.zip').extractall('/content/raw/heidata-MUXYRY')
    !python src/pilot_heidata_features.py --root /content/raw/heidata-MUXYRY --out "{HEI_OUT}" --seg classical

## 7. Ekstraksi fitur Illinois (semua sel sumur kontrol, 7 plate)

In [ ]:
IL_OUT = PROC / 'illinois/cells_controls_all.csv'
if IL_OUT.exists():
    print('sudah ada:', IL_OUT)
else:
    os.makedirs('/content/raw/illinois', exist_ok=True)
    for f in ('intensityTrajectories.zip', 'noiseTrajectories.zip'):
        shutil.copy(RAW / f, '/content/raw/illinois/')
    !python -m src.final.features illinois --root /content/raw/illinois --out "{IL_OUT}"

## 8. Ekstraksi fitur Essig (brightfield), satu percobaan per langkah
Zip disalin ke disk Colab (lebih cepat daripada membaca dari Drive), hasil + *checkpoint* langsung ke Drive, lalu zip lokal dihapus.
Sekitar 1 jam per 20 GB.

In [ ]:
ES_OUT = PROC / 'essig'
ES_OUT.mkdir(parents=True, exist_ok=True)
for e in ESSIG_EXPS:
    if (ES_OUT / f'{e}_features.csv').exists():
        print(e, 'sudah ada'); continue
    src = RAW / f'{e}.zip'
    if not src.exists():
        print(e, ': zip belum ada di Drive, dilewati'); continue
    t0 = time.time(); local = f'/content/{e}.zip'
    if not os.path.exists(local):
        print('salin', src, '->', local); shutil.copy(src, local)
    !python -I src/essig_exp_features.py --zip {local} --exp {e} --out-dir "{ES_OUT}"
    os.remove(local)
    print(e, 'selesai dalam %.0f menit' % ((time.time() - t0) / 60))
!ls -la "{ES_OUT}"

## 9. Latih dan evaluasi model final

In [ ]:
MODELS_DIR = DRIVE / 'models' / RUN_ID
for m in RUN_MODELS:
    cfg = f'configs/final/model_{m}.yaml'
    if m == 'A' and len(list(ES_OUT.glob('exp*_features.csv'))) < 2:
        print('Model A dilewati: perlu fitur Essig dari minimal 2 percobaan (validasi per percobaan)'); continue
    t0 = time.time()
    !python -m src.final.run --config {cfg} --data-root "{DRIVE}" --out results/final/model_{m} --models-dir "{MODELS_DIR}" --run-id {RUN_ID} {'--quick' if QUICK else ''}
    print(f'Model {m}: {(time.time() - t0) / 60:.1f} menit')
!python -m src.final.report --dir results/final

## 10. Simpan ke Drive dan push ke GitHub

In [ ]:
dst = DRIVE / 'results-colab' / RUN_ID
shutil.copytree('results/final', dst / 'final', dirs_exist_ok=True)
shutil.copy('/content/pip_freeze.txt', dst / 'pip_freeze.txt')
for m in RUN_MODELS:
    if os.path.isdir(f'results/final/model_{m}'):
        shutil.copy('/content/pip_freeze.txt', f'results/final/model_{m}/pip_freeze.txt')
print('Drive:', dst)

if PUSH_TO_GITHUB and not QUICK:
    big = [str(p) for p in Path('results/final').rglob('*') if p.is_file() and p.stat().st_size > 5 * 2**20]
    for p in big:
        print('tidak di-commit (> 5 MB, tetap di Drive):', p); os.remove(p)
    subprocess.run(['git', 'add', 'results/final'], check=True)
    msg = f'Hasil ML final run {RUN_ID} (Colab): model {", ".join(RUN_MODELS)}'
    if subprocess.run(['git', 'diff', '--cached', '--quiet']).returncode:
        subprocess.run(['git', 'commit', '-q', '-m', msg], check=True)
        print(git('pull', '--rebase', '-q', 'origin', 'main'))
        print(git('push', 'origin', 'HEAD:main'))
        print('Push ke GitHub selesai:', msg)
    else:
        print('Tidak ada perubahan hasil untuk di-commit')
else:
    print('Push dilewati (QUICK atau PUSH_TO_GITHUB = False)')

## 11. Setelah selesai
- Ringkasan tiga model: `results/final/README.md` (di GitHub) — sumber angka Bab IV.
- Rincian per model: `results/final/model_X/README.md`, gambar `confusion_matrix.png`, `roc.png`, `importance.png`.
- Model terlatih: Drive `models/<RUN_ID>/`; salinan lengkap run: Drive `results-colab/<RUN_ID>/`.
- Beri tahu Claude Code (Kai) bahwa run sudah selesai; hasil dibaca dari GitHub lalu dituliskan ke naskah dan Work Order.